# Output Visualization Notebook

This notebook reads already-generated CSV files from `outputs/` and creates comparison tables and plots. It does not call Neuronpedia.

It supports the current competence output files and older French `_fr.csv` expertise-style files by normalizing them into one shared table.


## 1. Setup


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display


if Path.cwd().name == "profession_concept_overlap":
    BASE_DIR = Path.cwd()
else:
    BASE_DIR = Path("profession_concept_overlap")

OUTPUT_DIR = BASE_DIR / "outputs"
EPSILON = 1e-9

GENDER_COLORS = {"control": "#b8b8b8", "male": "#8ecae6", "female": "#f4a6c1"}
DIVERGING_CMAP = sns.diverging_palette(330, 220, s=75, l=78, as_cmap=True)
LANGUAGE_ORDER = ["english", "simplified_chinese", "traditional_chinese", "french"]
GENDER_ORDER = ["control", "male", "female"]
OCCUPATION_ORDER = ["doctor", "engineer", "pilot", "teacher"]
PROFESSION_CANONICAL_MAP = {
    "doctor": "doctor",
    "médecin": "doctor",
    "engineer": "engineer",
    "spécialiste": "engineer",
    "pilot": "pilot",
    "pilote": "pilot",
    "teacher": "teacher",
    "professeur": "teacher",
}

sns.set_theme(style="whitegrid", context="notebook", palette="pastel")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)

OUTPUT_DIR


## 2. Load Output Files


In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)
    return pd.DataFrame()


def add_missing_language(frame: pd.DataFrame, language: str) -> pd.DataFrame:
    if frame.empty:
        return frame
    frame = frame.copy()
    if "language" not in frame.columns:
        frame["language"] = language
    elif frame["language"].isna().any():
        frame["language"] = frame["language"].fillna(language)
    return frame


trait_records = read_csv_if_exists(OUTPUT_DIR / "profession_gender_trait_fixed_feature_records.csv")
trait_group_summary = read_csv_if_exists(OUTPUT_DIR / "profession_gender_trait_group_summary.csv")
trait_feature_summary = read_csv_if_exists(OUTPUT_DIR / "profession_gender_trait_feature_summary_long.csv")

french_records = add_missing_language(
    read_csv_if_exists(OUTPUT_DIR / "profession_gender_expertise_fixed_feature_records_fr.csv"),
    "french",
)
french_group_summary = add_missing_language(
    read_csv_if_exists(OUTPUT_DIR / "profession_gender_expertise_group_summary_fr.csv"),
    "french",
)
french_feature_summary = add_missing_language(
    read_csv_if_exists(OUTPUT_DIR / "profession_gender_expertise_feature_summary_long_fr.csv"),
    "french",
)

for frame in [trait_records, trait_group_summary, trait_feature_summary, french_records, french_group_summary, french_feature_summary]:
    if not frame.empty and "trait_category" not in frame.columns:
        frame["trait_category"] = "competence"

print("trait_records", trait_records.shape)
print("trait_group_summary", trait_group_summary.shape)
print("french_records", french_records.shape)
print("french_group_summary", french_group_summary.shape)


## 3. Normalize And Combine


In [ ]:
def normalize_records(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    needed = [
        "language",
        "trait_category",
        "profession",
        "gender_group",
        "prompt",
        "prompt_pair",
        "feature",
        "source",
        "feature_index",
        "attribute_activation",
        "content_activation",
        "role_activation",
        "max_activation",
    ]
    normalized = frame.copy()
    for column in needed:
        if column not in normalized.columns:
            normalized[column] = pd.NA
    return normalized[needed]


def normalize_group_summary(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    needed = [
        "language",
        "trait_category",
        "profession",
        "gender_group",
        "prompt_count",
        "tested_feature_count",
        "mean_attribute_activation",
        "mean_content_activation",
        "mean_role_activation",
    ]
    normalized = frame.copy()
    for column in needed:
        if column not in normalized.columns:
            normalized[column] = pd.NA
    return normalized[needed]


activation_records = pd.concat(
    [normalize_records(trait_records), normalize_records(french_records)],
    ignore_index=True,
).dropna(subset=["language", "profession", "gender_group"], how="any")

group_summary = pd.concat(
    [normalize_group_summary(trait_group_summary), normalize_group_summary(french_group_summary)],
    ignore_index=True,
).dropna(subset=["language", "profession", "gender_group"], how="any")

# If a summary CSV is missing, derive group_summary from raw activation records.
if group_summary.empty and not activation_records.empty:
    group_summary = (
        activation_records
        .groupby(["language", "trait_category", "profession", "gender_group"], as_index=False)
        .agg(
            prompt_count=("prompt", "nunique"),
            tested_feature_count=("feature", "nunique"),
            mean_attribute_activation=("attribute_activation", "mean"),
            mean_content_activation=("content_activation", "mean"),
            mean_role_activation=("role_activation", "mean"),
        )
    )

for column in ["mean_attribute_activation", "mean_content_activation", "mean_role_activation"]:
    if column in group_summary.columns:
        group_summary[column] = pd.to_numeric(group_summary[column], errors="coerce")

activation_records["occupation"] = activation_records["profession"].map(PROFESSION_CANONICAL_MAP).fillna(activation_records["profession"].astype(str))
group_summary["occupation"] = group_summary["profession"].map(PROFESSION_CANONICAL_MAP).fillna(group_summary["profession"].astype(str))

activation_records["occupation"] = pd.Categorical(activation_records["occupation"], categories=OCCUPATION_ORDER, ordered=True)
group_summary["occupation"] = pd.Categorical(group_summary["occupation"], categories=OCCUPATION_ORDER, ordered=True)
group_summary["language"] = pd.Categorical(group_summary["language"], categories=LANGUAGE_ORDER, ordered=True)
group_summary["gender_group"] = pd.Categorical(group_summary["gender_group"], categories=GENDER_ORDER, ordered=True)
group_summary = group_summary.sort_values(["language", "occupation", "gender_group"]).reset_index(drop=True)

display(group_summary.head(20))
print("languages:", list(group_summary["language"].dropna().unique()))
print("occupations:", list(group_summary["occupation"].dropna().unique()))


## 4. Difference Tables


In [ ]:
def pct_relative_to_control(value: pd.Series, control: pd.Series) -> pd.Series:
    denominator = control.where(control.abs() > EPSILON)
    return ((value - control) / denominator) * 100


def signed_pct_difference(left: pd.Series, right: pd.Series) -> pd.Series:
    denominator = ((left.abs() + right.abs()) / 2).where(lambda s: s > EPSILON)
    return ((left - right) / denominator) * 100


group_difference_table = (
    group_summary
    .pivot_table(
        index=["language", "trait_category", "occupation"],
        columns="gender_group",
        values="mean_attribute_activation",
        fill_value=0.0,
        observed=False,
    )
    .reset_index()
)

for column in ["control", "male", "female"]:
    if column not in group_difference_table.columns:
        group_difference_table[column] = 0.0

group_difference_table["male_minus_control"] = group_difference_table["male"] - group_difference_table["control"]
group_difference_table["female_minus_control"] = group_difference_table["female"] - group_difference_table["control"]
group_difference_table["male_minus_female"] = group_difference_table["male"] - group_difference_table["female"]
group_difference_table["male_pct_vs_control"] = pct_relative_to_control(group_difference_table["male"], group_difference_table["control"])
group_difference_table["female_pct_vs_control"] = pct_relative_to_control(group_difference_table["female"], group_difference_table["control"])
group_difference_table["male_female_pct_gap"] = signed_pct_difference(group_difference_table["male"], group_difference_table["female"])

LANGUAGE_LABELS = {
    "english": "english",
    "traditional_chinese": "traditional chinese",
    "simplified_chinese": "simplified chinese",
    "french": "french",
}

male_female_comparison_table = group_difference_table[
    ["language", "occupation", "male", "female", "male_minus_female", "male_female_pct_gap"]
].copy()
male_female_comparison_table["language_label"] = male_female_comparison_table["language"].map(LANGUAGE_LABELS).fillna(male_female_comparison_table["language"].astype(str))
male_female_comparison_table["higher_group"] = "tie"
male_female_comparison_table.loc[male_female_comparison_table["male_female_pct_gap"] > 0, "higher_group"] = "male"
male_female_comparison_table.loc[male_female_comparison_table["male_female_pct_gap"] < 0, "higher_group"] = "female"
male_female_comparison_table = male_female_comparison_table[
    ["language", "language_label", "occupation", "higher_group", "male", "female", "male_minus_female", "male_female_pct_gap"]
].sort_values(["occupation", "language_label"])

display(group_difference_table)
display(male_female_comparison_table)


## 5. Styled Male-Female Comparison Table


In [ ]:
def format_signed_pct(value: float) -> str:
    if pd.isna(value):
        return ""
    return f"{value:+.1f}%"


def blend_rgb(start: tuple[int, int, int], end: tuple[int, int, int], weight: float) -> str:
    weight = max(0.0, min(1.0, float(weight)))
    rgb = [round(start_value + (end_value - start_value) * weight) for start_value, end_value in zip(start, end)]
    return f"rgb({rgb[0]}, {rgb[1]}, {rgb[2]})"


def style_signed_percentage_table(frame: pd.DataFrame, pct_columns: list[str]):
    max_abs = frame[pct_columns].abs().max().max()
    max_abs = float(max_abs) if pd.notna(max_abs) and max_abs > EPSILON else 1.0
    white = (255, 255, 255)
    male_blue = (142, 202, 230)
    female_pink = (244, 166, 193)

    def color_signed_pct(value):
        if pd.isna(value):
            return ""
        weight = min(abs(float(value)) / max_abs, 1.0)
        if value > 0:
            background = blend_rgb(white, male_blue, weight)
        elif value < 0:
            background = blend_rgb(white, female_pink, weight)
        else:
            background = "white"
        return f"background-color: {background}; color: #111111; font-weight: 600"

    return (
        frame.style
        .format({column: format_signed_pct for column in pct_columns})
        .format({"male": "{:.3f}", "female": "{:.3f}", "male_minus_female": "{:+.3f}"})
        .map(color_signed_pct, subset=pct_columns)
        .set_properties(subset=pct_columns, **{"text-align": "center"})
    )


language_pct_columns = ["english", "simplified chinese", "traditional chinese", "french"]
occupation_language_pct_table = (
    male_female_comparison_table
    .pivot_table(
        index="occupation",
        columns="language_label",
        values="male_female_pct_gap",
        aggfunc="mean",
        observed=False,
    )
    .reindex(columns=language_pct_columns)
    .reset_index()
)

occupation_language_winner_table = (
    male_female_comparison_table
    .pivot_table(
        index="occupation",
        columns="language_label",
        values="higher_group",
        aggfunc=lambda values: ", ".join(sorted(set(str(value) for value in values))),
        observed=False,
    )
    .reindex(columns=language_pct_columns)
    .reset_index()
)

occupation_language_pct_table.to_csv(OUTPUT_DIR / "occupation_language_male_female_pct_table.csv", index=False)
occupation_language_winner_table.to_csv(OUTPUT_DIR / "occupation_language_male_female_winner_table.csv", index=False)

display(style_signed_percentage_table(occupation_language_pct_table, language_pct_columns))
display(occupation_language_winner_table)
display(style_signed_percentage_table(male_female_comparison_table, ["male_female_pct_gap"]))


## 6. Visualization Gallery


### 1. Mean Activation By Language, Occupation, And Gender


In [ ]:
plot_data = group_summary.dropna(subset=["mean_attribute_activation"]).copy()
grid = sns.catplot(
    data=plot_data,
    x="occupation",
    y="mean_attribute_activation",
    hue="gender_group",
    col="language",
    kind="bar",
    hue_order=GENDER_ORDER,
    palette=GENDER_COLORS,
    height=4,
    aspect=1.15,
)
grid.set_axis_labels("", "mean attribute-token activation")
grid.set_titles("{col_name}")
for ax in grid.axes.flat:
    ax.tick_params(axis="x", rotation=20)
plt.show()


### 1b. Normalized Mean Activation By Language, Occupation, And Gender


In [ ]:
normalized_plot_data = group_summary.dropna(subset=["mean_attribute_activation"]).copy()

language_stats = normalized_plot_data.groupby("language", observed=True)["mean_attribute_activation"].agg(["mean", "std"]).reset_index()
normalized_plot_data = normalized_plot_data.merge(language_stats, on="language", how="left")
normalized_plot_data["language_z_activation"] = (
    (normalized_plot_data["mean_attribute_activation"] - normalized_plot_data["mean"])
    / normalized_plot_data["std"].where(normalized_plot_data["std"] > EPSILON)
)

grid = sns.catplot(
    data=normalized_plot_data,
    x="occupation",
    y="language_z_activation",
    hue="gender_group",
    col="language",
    kind="bar",
    hue_order=GENDER_ORDER,
    palette=GENDER_COLORS,
    height=4,
    aspect=1.15,
)
for ax in grid.axes.flat:
    ax.axhline(0, color="#333333", linewidth=0.9)
    ax.tick_params(axis="x", rotation=20)
grid.set_axis_labels("", "language-normalized activation z-score")
grid.set_titles("{col_name}")
plt.show()


### 2. Signed Male-Female Percentage Gap


In [ ]:
plot_data = male_female_comparison_table.copy()
plot_data["label"] = plot_data["language"].astype(str) + " | " + plot_data["occupation"].astype(str)
plot_data = plot_data.sort_values("male_female_pct_gap")
colors = [GENDER_COLORS["male"] if value >= 0 else GENDER_COLORS["female"] for value in plot_data["male_female_pct_gap"]]

fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(plot_data))), constrained_layout=True)
ax.barh(plot_data["label"], plot_data["male_female_pct_gap"], color=colors)
ax.axvline(0, color="#333333", linewidth=0.9)
ax.set_title("Male vs female competence activation gap")
ax.set_xlabel("signed % gap: positive = male higher, negative = female higher")
ax.set_ylabel("")
plt.show()


### 3. Language-Occupation Signed Gap Matrix


In [ ]:
matrix = (
    male_female_comparison_table
    .pivot_table(
        index="language_label",
        columns="occupation",
        values="male_female_pct_gap",
        observed=True,
    )
    .reindex(index=["english", "simplified chinese", "traditional chinese", "french"], columns=OCCUPATION_ORDER)
)

max_abs = max(float(matrix.abs().max().max()), EPSILON)
fig, ax = plt.subplots(figsize=(8.5, max(3.2, 0.8 * len(matrix))), constrained_layout=True)
sns.heatmap(
    matrix,
    annot=True,
    fmt="+.1f",
    cmap=DIVERGING_CMAP,
    center=0,
    vmin=-max_abs,
    vmax=max_abs,
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"label": "signed % gap"},
    ax=ax,
)
ax.set_title("Signed male-female percentage gap by language and occupation")
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticklabels([label.title() for label in matrix.columns], rotation=0)
ax.set_yticklabels(["English", "Simplified Chinese", "Traditional Chinese", "French"], rotation=0)
plt.show()


### 4. Male And Female Change Relative To Control


In [ ]:
control_long = group_difference_table.melt(
    id_vars=["language", "occupation"],
    value_vars=["male_pct_vs_control", "female_pct_vs_control"],
    var_name="comparison",
    value_name="pct_vs_control",
).dropna(subset=["pct_vs_control"])
control_long["gender"] = control_long["comparison"].str.replace("_pct_vs_control", "", regex=False)

grid = sns.catplot(
    data=control_long,
    x="occupation",
    y="pct_vs_control",
    hue="gender",
    col="language",
    kind="bar",
    palette={"male": GENDER_COLORS["male"], "female": GENDER_COLORS["female"]},
    height=4,
    aspect=1.15,
)
for ax in grid.axes.flat:
    ax.axhline(0, color="#333333", linewidth=0.9)
    ax.tick_params(axis="x", rotation=20)
grid.set_axis_labels("", "% change relative to control")
grid.set_titles("{col_name}")
plt.show()


### 5. Male Versus Female Scatter


In [ ]:
plot_data = male_female_comparison_table.copy()
fig, ax = plt.subplots(figsize=(6.5, 6), constrained_layout=True)
sns.scatterplot(
    data=plot_data,
    x="female",
    y="male",
    hue="language",
    style="occupation",
    s=110,
    ax=ax,
)
limit_min = min(plot_data["female"].min(), plot_data["male"].min())
limit_max = max(plot_data["female"].max(), plot_data["male"].max())
ax.plot([limit_min, limit_max], [limit_min, limit_max], color="#333333", linestyle="--", linewidth=1)
ax.set_title("Male vs female mean activation")
ax.set_xlabel("female mean activation")
ax.set_ylabel("male mean activation")
plt.show()


### 6. Activation Distribution By Gender


In [ ]:
if activation_records.empty:
    print("No raw activation records available.")
else:
    distribution_data = activation_records.dropna(subset=["attribute_activation"]).copy()
    distribution_data["gender_group"] = pd.Categorical(distribution_data["gender_group"], categories=GENDER_ORDER, ordered=True)
    grid = sns.catplot(
        data=distribution_data,
        x="gender_group",
        y="attribute_activation",
        hue="gender_group",
        col="language",
        kind="boxen",
        palette=GENDER_COLORS,
        height=4,
        aspect=1.05,
        legend=False,
    )
    grid.set_axis_labels("", "prompt-feature activation")
    grid.set_titles("{col_name}")
    plt.show()


### 7. Occupation-Level Gender Gap Ranking By Language


In [ ]:
plot_data = male_female_comparison_table.sort_values(["language", "male_female_pct_gap"]).copy()
grid = sns.FacetGrid(plot_data, col="language", height=4, aspect=1.1, sharex=False)

def draw_ranked_bars(data, **kwargs):
    colors = [GENDER_COLORS["male"] if value >= 0 else GENDER_COLORS["female"] for value in data["male_female_pct_gap"]]
    plt.barh(data["occupation"], data["male_female_pct_gap"], color=colors)
    plt.axvline(0, color="#333333", linewidth=0.9)

grid.map_dataframe(draw_ranked_bars)
grid.set_axis_labels("signed % gap", "")
grid.set_titles("{col_name}")
plt.show()


### 8. Feature-Level Top Gender Gaps


In [ ]:
if activation_records.empty:
    print("No raw activation records available.")
else:
    feature_summary = (
        activation_records
        .dropna(subset=["language", "occupation"])
        .groupby(["language", "occupation", "feature", "gender_group"], as_index=False, observed=True)
        .agg(mean_attribute_activation=("attribute_activation", "mean"))
    )
    feature_diff = (
        feature_summary
        .pivot_table(
            index=["language", "occupation", "feature"],
            columns="gender_group",
            values="mean_attribute_activation",
            fill_value=0.0,
            observed=True,
        )
        .reset_index()
    )
    for column in ["male", "female"]:
        if column not in feature_diff.columns:
            feature_diff[column] = 0.0
    feature_diff["male_minus_female"] = feature_diff["male"] - feature_diff["female"]
    feature_diff["abs_gap"] = feature_diff["male_minus_female"].abs()

    top_features = feature_diff.sort_values("abs_gap", ascending=False).head(20).copy()
    top_features["label"] = top_features["language"].astype(str) + " | " + top_features["occupation"].astype(str) + " | " + top_features["feature"].astype(str)
    colors = [GENDER_COLORS["male"] if value >= 0 else GENDER_COLORS["female"] for value in top_features["male_minus_female"]]

    fig, ax = plt.subplots(figsize=(10, max(5, 0.38 * len(top_features))), constrained_layout=True)
    ax.barh(top_features["label"], top_features["male_minus_female"], color=colors)
    ax.axvline(0, color="#333333", linewidth=0.9)
    ax.invert_yaxis()
    ax.set_title("Top feature-level male-female activation gaps")
    ax.set_xlabel("male minus female mean activation")
    ax.set_ylabel("")
    plt.show()


### 9. Prompt-Level Matched Male-Female Gaps


In [ ]:
if activation_records.empty or "prompt_pair" not in activation_records.columns:
    print("No prompt-pair activation records available.")
else:
    prompt_summary = (
        activation_records[activation_records["gender_group"].isin(["male", "female"])]
        .dropna(subset=["language", "occupation"])
        .groupby(["language", "occupation", "prompt_pair", "gender_group"], as_index=False, observed=True)
        .agg(mean_attribute_activation=("attribute_activation", "mean"))
    )
    prompt_diff = (
        prompt_summary
        .pivot_table(
            index=["language", "occupation", "prompt_pair"],
            columns="gender_group",
            values="mean_attribute_activation",
            fill_value=0.0,
            observed=True,
        )
        .reset_index()
    )
    for column in ["male", "female"]:
        if column not in prompt_diff.columns:
            prompt_diff[column] = 0.0
    prompt_diff["male_minus_female"] = prompt_diff["male"] - prompt_diff["female"]
    prompt_diff["label"] = prompt_diff["language"].astype(str) + " | " + prompt_diff["occupation"].astype(str) + " | " + prompt_diff["prompt_pair"].astype(str)
    prompt_diff = prompt_diff.sort_values("male_minus_female")
    colors = [GENDER_COLORS["male"] if value >= 0 else GENDER_COLORS["female"] for value in prompt_diff["male_minus_female"]]

    fig, ax = plt.subplots(figsize=(10, max(5, 0.26 * len(prompt_diff))), constrained_layout=True)
    ax.barh(prompt_diff["label"], prompt_diff["male_minus_female"], color=colors)
    ax.axvline(0, color="#333333", linewidth=0.9)
    ax.set_title("Prompt-level male-female activation gaps")
    ax.set_xlabel("male minus female mean activation")
    ax.set_ylabel("")
    plt.show()


### 10. Data Coverage By Language And Occupation


In [ ]:
if activation_records.empty:
    print("No raw activation records available.")
else:
    coverage = (
        activation_records
        .dropna(subset=["language", "occupation"])
        .groupby(["language", "occupation", "gender_group"], as_index=False, observed=True)
        .agg(record_count=("attribute_activation", "size"))
    )
    grid = sns.catplot(
        data=coverage,
        x="occupation",
        y="record_count",
        hue="gender_group",
        col="language",
        kind="bar",
        hue_order=GENDER_ORDER,
        palette=GENDER_COLORS,
        height=4,
        aspect=1.15,
    )
    grid.set_axis_labels("", "activation record count")
    grid.set_titles("{col_name}")
    for ax in grid.axes.flat:
        ax.tick_params(axis="x", rotation=20)
    plt.show()


### 11. Cross-Language Occupation Trajectories


In [ ]:
with sns.axes_style("darkgrid"):
    trajectory_data = male_female_comparison_table.dropna(subset=["male_female_pct_gap"]).copy()
    trajectory_data["language_label"] = pd.Categorical(
        trajectory_data["language_label"],
        categories=["english", "chinese", "french"],
        ordered=True,
    )
    fig, ax = plt.subplots(figsize=(8.5, 4.5), constrained_layout=True)
    sns.lineplot(
        data=trajectory_data,
        x="language_label",
        y="male_female_pct_gap",
        hue="occupation",
        marker="o",
        linewidth=2,
        ax=ax,
    )
    ax.axhline(0, color="#333333", linewidth=0.9)
    ax.set_title("How each occupation's male-female gap changes across languages")
    ax.set_xlabel("")
    ax.set_ylabel("signed % gap")
    plt.show()


### 12. Male And Female Activation Dot Plot


In [ ]:
with sns.axes_style("white"):
    dot_data = male_female_comparison_table.melt(
        id_vars=["language_label", "occupation"],
        value_vars=["male", "female"],
        var_name="gender",
        value_name="mean_activation",
    )
    dot_data["language_occupation"] = dot_data["language_label"].astype(str) + " | " + dot_data["occupation"].astype(str)
    dot_data = dot_data.sort_values(["language_label", "occupation", "gender"])
    fig, ax = plt.subplots(figsize=(8.5, max(4.5, 0.42 * dot_data["language_occupation"].nunique())), constrained_layout=True)
    sns.stripplot(
        data=dot_data,
        y="language_occupation",
        x="mean_activation",
        hue="gender",
        dodge=True,
        size=8,
        palette={"male": GENDER_COLORS["male"], "female": GENDER_COLORS["female"]},
        ax=ax,
    )
    ax.set_title("Male and female mean activations for each language-occupation pair")
    ax.set_xlabel("mean activation")
    ax.set_ylabel("")
    plt.show()


### 13. Feature-Level Gap Distribution


In [ ]:
with sns.axes_style("dark"):
    if activation_records.empty:
        print("No raw activation records available.")
    else:
        feature_distribution = (
            activation_records
            .dropna(subset=["language", "occupation"])
            .groupby(["language", "occupation", "feature", "gender_group"], as_index=False, observed=True)
            .agg(mean_attribute_activation=("attribute_activation", "mean"))
            .pivot_table(
                index=["language", "occupation", "feature"],
                columns="gender_group",
                values="mean_attribute_activation",
                observed=True,
            )
            .reset_index()
        )
        for column in ["male", "female"]:
            if column not in feature_distribution.columns:
                feature_distribution[column] = 0.0
        feature_distribution = feature_distribution.dropna(subset=["male", "female"], how="all")
        feature_distribution["male_minus_female"] = feature_distribution["male"].fillna(0) - feature_distribution["female"].fillna(0)

        fig, ax = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
        sns.boxplot(
            data=feature_distribution,
            x="language",
            y="male_minus_female",
            hue="language",
            palette="pastel",
            ax=ax,
            legend=False,
        )
        ax.axhline(0, color="#333333", linewidth=0.9)
        ax.set_title("Distribution of feature-level male-female gaps by language")
        ax.set_xlabel("")
        ax.set_ylabel("male minus female activation")
        plt.show()


### 14. Prompt-Pair Gap Small Multiples


In [ ]:
with sns.axes_style("ticks"):
    if activation_records.empty or "prompt_pair" not in activation_records.columns:
        print("No prompt-pair activation records available.")
    else:
        prompt_gap_data = (
            activation_records[activation_records["gender_group"].isin(["male", "female"])]
            .dropna(subset=["language", "occupation", "prompt_pair"])
            .groupby(["language", "occupation", "prompt_pair", "gender_group"], as_index=False, observed=True)
            .agg(mean_attribute_activation=("attribute_activation", "mean"))
            .pivot_table(
                index=["language", "occupation", "prompt_pair"],
                columns="gender_group",
                values="mean_attribute_activation",
                observed=True,
            )
            .reset_index()
        )
        for column in ["male", "female"]:
            if column not in prompt_gap_data.columns:
                prompt_gap_data[column] = 0.0
        prompt_gap_data = prompt_gap_data.dropna(subset=["male", "female"], how="all")
        prompt_gap_data["male_minus_female"] = prompt_gap_data["male"].fillna(0) - prompt_gap_data["female"].fillna(0)

        grid = sns.catplot(
            data=prompt_gap_data,
            x="occupation",
            y="male_minus_female",
            hue="occupation",
            col="language",
            kind="point",
            errorbar=None,
            height=4,
            aspect=1.05,
            palette="pastel",
            legend=False,
        )
        for ax in grid.axes.flat:
            ax.axhline(0, color="#333333", linewidth=0.9)
            ax.tick_params(axis="x", rotation=20)
        grid.set_axis_labels("", "prompt-pair male minus female")
        grid.set_titles("{col_name}")
        plt.show()


### 15. Control-Relative Gender Shift Scatter


In [ ]:
with sns.axes_style("whitegrid"):
    shift_data = group_difference_table.dropna(subset=["male_pct_vs_control", "female_pct_vs_control"]).copy()
    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    sns.scatterplot(
        data=shift_data,
        x="female_pct_vs_control",
        y="male_pct_vs_control",
        hue="language",
        style="occupation",
        s=120,
        ax=ax,
    )
    limit_values = [
        shift_data["female_pct_vs_control"].min(),
        shift_data["female_pct_vs_control"].max(),
        shift_data["male_pct_vs_control"].min(),
        shift_data["male_pct_vs_control"].max(),
    ]
    limit_min = min(limit_values)
    limit_max = max(limit_values)
    ax.plot([limit_min, limit_max], [limit_min, limit_max], color="#333333", linestyle="--", linewidth=1)
    ax.axhline(0, color="#999999", linewidth=0.8)
    ax.axvline(0, color="#999999", linewidth=0.8)
    ax.set_title("Control-relative activation shifts")
    ax.set_xlabel("female % change vs control")
    ax.set_ylabel("male % change vs control")
    plt.show()


## 9. Shared Multilingual Feature Analysis


### 16. Shared Active Feature Counts


In [ ]:
SHARED_FEATURE_ACTIVATION_THRESHOLD = 0.0
REQUIRED_SHARED_LANGUAGES = ["english", "simplified_chinese", "traditional_chinese", "french"]

if activation_records.empty:
    print("No raw activation records available.")
    shared_feature_language_table = pd.DataFrame()
    shared_features = pd.DataFrame()
else:
    feature_language_activity = (
        activation_records
        .dropna(subset=["language", "occupation", "feature"])
        .groupby(["occupation", "feature", "language"], as_index=False, observed=True)
        .agg(
            max_activation=("attribute_activation", "max"),
            mean_activation=("attribute_activation", "mean"),
            active_prompt_feature_count=("attribute_activation", lambda values: int((values > SHARED_FEATURE_ACTIVATION_THRESHOLD).sum())),
        )
    )
    feature_language_activity["is_active"] = feature_language_activity["max_activation"] > SHARED_FEATURE_ACTIVATION_THRESHOLD

    active_feature_counts_by_language = (
        feature_language_activity[feature_language_activity["is_active"]]
        .groupby(["language", "occupation"], as_index=False, observed=True)
        .agg(active_feature_count=("feature", "nunique"))
    )
    active_feature_counts_by_language["language_label"] = active_feature_counts_by_language["language"].map(LANGUAGE_LABELS).fillna(
        active_feature_counts_by_language["language"].astype(str)
    )
    active_feature_counts_by_language["occupation"] = pd.Categorical(
        active_feature_counts_by_language["occupation"],
        categories=OCCUPATION_ORDER,
        ordered=True,
    )
    active_feature_counts_by_language.to_csv(OUTPUT_DIR / "active_feature_counts_by_language_occupation.csv", index=False)

    display(active_feature_counts_by_language.sort_values(["language_label", "occupation"]))

    fig, ax = plt.subplots(figsize=(8.5, 4.2), constrained_layout=True)
    sns.barplot(
        data=active_feature_counts_by_language,
        x="occupation",
        y="active_feature_count",
        hue="language_label",
        palette="pastel",
        ax=ax,
    )
    ax.set_title("Non-zero active features by language and occupation")
    ax.set_xlabel("")
    ax.set_ylabel("active feature count")
    ax.set_ylim(0, FEATURE_LIMIT_PER_TRAIT if "FEATURE_LIMIT_PER_TRAIT" in globals() else 20)
    ax.legend(title="")
    plt.show()

    shared_feature_language_table = (
        feature_language_activity[feature_language_activity["is_active"]]
        .pivot_table(
            index=["occupation", "feature"],
            columns="language",
            values="max_activation",
            aggfunc="max",
            observed=True,
        )
        .reset_index()
    )

    for language in REQUIRED_SHARED_LANGUAGES:
        if language not in shared_feature_language_table.columns:
            shared_feature_language_table[language] = pd.NA

    shared_feature_language_table["language_count"] = shared_feature_language_table[REQUIRED_SHARED_LANGUAGES].notna().sum(axis=1)
    shared_feature_language_table["mean_max_activation"] = shared_feature_language_table[REQUIRED_SHARED_LANGUAGES].mean(axis=1)
    shared_features = shared_feature_language_table[shared_feature_language_table["language_count"] == len(REQUIRED_SHARED_LANGUAGES)].copy()
    shared_features = shared_features.sort_values(["occupation", "mean_max_activation"], ascending=[True, False])

    shared_counts = (
        shared_features
        .groupby("occupation", as_index=False, observed=True)
        .agg(shared_feature_count=("feature", "nunique"))
        .sort_values("occupation")
    )

    display(shared_counts)
    display(shared_features.head(30))

    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    sns.barplot(data=shared_counts, x="occupation", y="shared_feature_count", hue="occupation", palette="pastel", legend=False, ax=ax)
    ax.set_title("Features active in all four languages by occupation")
    ax.set_xlabel("")
    ax.set_ylabel("shared active feature count")
    plt.show()


### 17. Top Shared Feature Activations Across Languages


In [ ]:
if activation_records.empty or shared_features.empty:
    print("No shared multilingual features available.")
else:
    TOP_SHARED_FEATURES = 16
    top_shared_features = shared_features.head(TOP_SHARED_FEATURES)[["occupation", "feature"]]
    top_shared_activity = feature_language_activity.merge(top_shared_features, on=["occupation", "feature"], how="inner")
    top_shared_activity["label"] = top_shared_activity["occupation"].astype(str) + " | " + top_shared_activity["feature"].astype(str)

    fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * top_shared_activity["label"].nunique())), constrained_layout=True)
    sns.barplot(
        data=top_shared_activity,
        y="label",
        x="max_activation",
        hue="language",
        palette="Set2",
        ax=ax,
    )
    ax.set_title("Top shared features: max activation by language")
    ax.set_xlabel("max attribute activation")
    ax.set_ylabel("")
    plt.show()


### 18. Shared Feature Male-Female Bias


In [ ]:
if activation_records.empty or shared_features.empty:
    print("No shared multilingual features available.")
else:
    shared_feature_gender = (
        activation_records
        .merge(shared_features[["occupation", "feature"]], on=["occupation", "feature"], how="inner")
        .query("gender_group in ['male', 'female']")
        .groupby(["occupation", "feature", "language", "gender_group"], as_index=False, observed=True)
        .agg(mean_activation=("attribute_activation", "mean"))
        .pivot_table(
            index=["occupation", "feature", "language"],
            columns="gender_group",
            values="mean_activation",
            observed=True,
        )
        .reset_index()
    )
    for column in ["male", "female"]:
        if column not in shared_feature_gender.columns:
            shared_feature_gender[column] = 0.0
    shared_feature_gender = shared_feature_gender.dropna(subset=["male", "female"], how="all")
    shared_feature_gender["male_minus_female"] = shared_feature_gender["male"].fillna(0) - shared_feature_gender["female"].fillna(0)
    shared_feature_gender["abs_gap"] = shared_feature_gender["male_minus_female"].abs()

    top_shared_bias = shared_feature_gender.sort_values("abs_gap", ascending=False).head(24).copy()
    top_shared_bias["label"] = top_shared_bias["language"].astype(str) + " | " + top_shared_bias["occupation"].astype(str) + " | " + top_shared_bias["feature"].astype(str)
    colors = [GENDER_COLORS["male"] if value >= 0 else GENDER_COLORS["female"] for value in top_shared_bias["male_minus_female"]]

    fig, ax = plt.subplots(figsize=(11, max(6, 0.35 * len(top_shared_bias))), constrained_layout=True)
    ax.barh(top_shared_bias["label"], top_shared_bias["male_minus_female"], color=colors)
    ax.axvline(0, color="#333333", linewidth=0.9)
    ax.invert_yaxis()
    ax.set_title("Largest male-female gaps among features shared across languages")
    ax.set_xlabel("male minus female mean activation")
    ax.set_ylabel("")
    plt.show()


### 19. Shared Feature Bias Matrix


In [ ]:
if activation_records.empty or shared_features.empty:
    print("No shared multilingual features available.")
else:
    matrix_source = shared_feature_gender.copy()
    top_matrix_features = (
        matrix_source
        .groupby(["occupation", "feature"], as_index=False, observed=True)
        .agg(mean_abs_gap=("abs_gap", "mean"))
        .sort_values("mean_abs_gap", ascending=False)
        .head(12)
    )
    matrix_source = matrix_source.merge(top_matrix_features[["occupation", "feature"]], on=["occupation", "feature"], how="inner")
    matrix_source["row"] = matrix_source["occupation"].astype(str) + " | " + matrix_source["feature"].astype(str)
    bias_matrix = matrix_source.pivot_table(
        index="row",
        columns="language",
        values="male_minus_female",
        observed=True,
    )
    max_abs = max(float(bias_matrix.abs().max().max()), EPSILON)
    fig, ax = plt.subplots(figsize=(8, max(5, 0.4 * len(bias_matrix))), constrained_layout=True)
    sns.heatmap(
        bias_matrix,
        annot=True,
        fmt="+.2f",
        cmap=DIVERGING_CMAP,
        center=0,
        vmin=-max_abs,
        vmax=max_abs,
        linewidths=0.8,
        linecolor="white",
        cbar_kws={"label": "male minus female activation"},
        ax=ax,
    )
    ax.set_title("Shared feature male-female bias across languages")
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.show()


### 20. Top Feature Activations By Gender Group


In [ ]:
TOP_FEATURE_GRID_COUNT = 12


def short_feature_label(feature: str, max_chars: int = 28) -> str:
    text = str(feature)
    if "/" in text:
        source, index = text.rsplit("/", 1)
        layer = source.split("-")[0]
        text = f"{layer}/{index}"
    if len(text) > max_chars:
        return text[: max_chars - 3] + "..."
    return text


if activation_records.empty:
    print("No raw activation records available.")
else:
    feature_gender_summary = (
        activation_records
        .dropna(subset=["language", "occupation", "feature", "gender_group"])
        .groupby(["language", "occupation", "feature", "gender_group"], as_index=False, observed=True)
        .agg(mean_attribute_activation=("attribute_activation", "mean"))
    )

    feature_gender_wide = (
        feature_gender_summary
        .pivot_table(
            index=["language", "occupation", "feature"],
            columns="gender_group",
            values="mean_attribute_activation",
            fill_value=0.0,
            observed=True,
        )
        .reset_index()
    )

    for column in GENDER_ORDER:
        if column not in feature_gender_wide.columns:
            feature_gender_wide[column] = 0.0

    feature_gender_wide["spread"] = feature_gender_wide[GENDER_ORDER].max(axis=1) - feature_gender_wide[GENDER_ORDER].min(axis=1)
    feature_gender_wide.to_csv(OUTPUT_DIR / "feature_gender_activation_spreads.csv", index=False)

    for language in LANGUAGE_ORDER:
        language_frame = feature_gender_wide[feature_gender_wide["language"].astype(str) == language]
        if language_frame.empty:
            continue

        fig, axes = plt.subplots(2, 2, figsize=(16, 13), constrained_layout=True)
        axes = axes.flatten()

        for ax, occupation in zip(axes, OCCUPATION_ORDER):
            occupation_frame = (
                language_frame[language_frame["occupation"].astype(str) == occupation]
                .sort_values("spread", ascending=False)
                .head(TOP_FEATURE_GRID_COUNT)
                .copy()
            )

            if occupation_frame.empty:
                ax.set_title(occupation.title())
                ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue

            occupation_frame = occupation_frame.iloc[::-1].reset_index(drop=True)
            y_positions = range(len(occupation_frame))
            bar_height = 0.24
            offsets = {"control": -bar_height, "male": 0.0, "female": bar_height}

            for gender_group in GENDER_ORDER:
                ax.barh(
                    [position + offsets[gender_group] for position in y_positions],
                    occupation_frame[gender_group],
                    height=bar_height,
                    color=GENDER_COLORS[gender_group],
                    label=gender_group,
                )

            ax.set_yticks(list(y_positions))
            ax.set_yticklabels([short_feature_label(feature) for feature in occupation_frame["feature"]], fontsize=8)
            ax.set_title(occupation.title())
            ax.set_xlabel("mean attribute activation")
            ax.grid(axis="x", alpha=0.25)

        for ax in axes[len(OCCUPATION_ORDER):]:
            ax.set_axis_off()

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper center", ncol=3, title="gender group")
        fig.suptitle(f"{language}: top {TOP_FEATURE_GRID_COUNT} feature activations by gender group", y=1.02, fontsize=15)
        plt.show()
